In [ ]:
from dataclasses import dataclass
from datasets import load_dataset
from torchvision import transforms
import torch
from accelerate import Accelerator
from huggingface_hub import create_repo, upload_folder
from tqdm.auto import tqdm
from pathlib import Path
import os
from accelerate import notebook_launcher
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler, DDPMPipeline


Variable definiton

In [ ]:
@dataclass
class TrainingConfig:
    image_size = 128  # the generated image resolution
    train_batch_size = 16
    eval_batch_size = 16  # how many images to sample during evaluation
    num_epochs = 50
    gradient_accumulation_steps = 1
    learning_rate = 1e-4
    lr_warmup_steps = 500
    save_image_epochs = 10
    save_model_epochs = 30
    mixed_precision = "fp16"  # `no` for float32, `fp16` for automatic mixed precision
    output_dir = "ddpm-butterflies-128"  # the model name locally and on the HF Hub

    push_to_hub = False # whether to upload the saved model to the HF Hub
    hub_model_id = "<your-username>/<my-awesome-model>"  # the name of the repository to create on the HF Hub
    hub_private_repo = None
    overwrite_output_dir = True  # overwrite the old model when re-running the notebook
    seed = 0

# 1. Initialize Config FIRST
config = TrainingConfig()

# 2. Dataset Selection
config.dataset_name = "huggan/smithsonian_butterflies_subset"
dataset = load_dataset(config.dataset_name, split="train")

# dataset selction
dataset = load_dataset("huggan/smithsonian_butterflies_subset", split="train")



image preporcessing I

In [ ]:
preprocess = transforms.Compose(
    [
        transforms.Resize((TrainingConfig.image_size, TrainingConfig.image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
)


def transform(examples):
    images = [preprocess(image.convert("RGB")) for image in examples["image"]]
    return {"images": images}

# Apply the transform to the dataset
dataset.set_transform(transform)